# Inspect the receipt-time forecasting evidence

This notebook reads stored artifacts. The example is entirely synthetic; it is not paper performance evidence. Final result tables are read only after analysis is complete. Run from the repository root or this notebook directory.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
example = root / "runs/clean_environment_v4/clean_example"
if not example.exists():
    raise FileNotFoundError("Run the synthetic example and set example to its output directory first.")
manifest = json.loads((example / "manifest.json").read_text())
print(manifest["data_kind"])
print(pd.read_csv(example / "metrics.csv").to_string(index=False))


## Visible state around restoration

Measurement age can remain positive despite recent transport receipts; restored live measurements can coexist with missing history. These are observer features, not the hidden interruption plan.


In [ ]:
features = pd.read_parquet(example / "visible_features.parquet")
start = pd.Timestamp(manifest["fault"]["start"])
window = features.loc[start-pd.Timedelta(hours=6):start+pd.Timedelta(hours=36)]
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
window[["pv_age", "irradiance_age"]].plot(ax=axes[0])
window[["pv_missing_fraction", "irradiance_missing_fraction"]].plot(ax=axes[1])
axes[0].set_ylabel("Hours old")
axes[1].set_ylabel("Missing slot fraction")
fig.tight_layout()
plt.show()


## Issued quantiles

The actual target is an evaluator column. The online forecaster and calibrator do not receive it until its simulated delivery. Forecasts are plotted at their target interval end.


In [ ]:
forecasts = pd.read_parquet(example / "forecasts.parquet")
part = forecasts.loc[forecasts.method.eq("physical_recovery") & forecasts.horizon.eq(1)]
part = part.loc[part.origin.between(start-pd.Timedelta(hours=6), start+pd.Timedelta(hours=36))]
fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(part.target_end, part.q05, part.q95, alpha=.2, label="Issued 90% interval")
ax.plot(part.target_end, part.q50, label="Issued median")
ax.plot(part.target_end, part.actual_kw, color="black", label="Synthetic evaluator target")
ax.set_ylabel("kW")
ax.legend()
fig.tight_layout()
plt.show()


## Final fixed-site evidence, when complete

The primary contrast is recovery minus availability NWIS. Positive values mean worse forecasts. Calendar-block intervals keep repeated weather together; they are not population-level guarantees.


In [ ]:
analysis = root / "runs/final_2017_v4/analysis"
if not (analysis / "manifest.json").exists():
    print("Final analysis is not complete; no result is asserted.")
else:
    contrasts = pd.read_csv(analysis / "Contrasts.csv")
    selected = contrasts.loc[contrasts.population.eq("recorded_ac") & contrasts.pool.eq("faulted_primary") & contrasts.block_days.eq(7) & contrasts.contrast.eq("physical_recovery_minus_physical_availability")]
    print(selected[["site", "endpoint", "difference", "ci_lower", "ci_upper", "paired_rows", "calendar_blocks"]].to_string(index=False))
